# StudyMate · Part 6 of 8: Final Exam Real Questions

**Assignment part:** Evaluation (final exam / holdout)  
**Group 7:** Sadia Saeed · Eva Abouabouharb · Neida Guzman

A fair final test on **real Data Science Stack Exchange questions** that nobody on the team wrote: sample 120 questions, label them (53 labeled, 67 outside the library), and run the final exam **once**, comparing StudyMate with the hybrid-only baseline.

> 📖 **This notebook is for reading.** It is one section of the full notebook, with the real outputs kept. Each part uses results from the parts before it, so to **run** StudyMate, open [`FN_StudyMate_Group7_ITAI2377.ipynb`](FN_StudyMate_Group7_ITAI2377.ipynb) and use **Runtime → Run all**.

## 8. Final Test, Part 1: Pick Real Questions for Labeling

**Purpose:** Build a fair final test from real questions asked by people on Data Science Stack Exchange.

**Data source:** Hugging Face `mgp123/datascience-stackexchange-with-similar-questions` (Data Science Stack Exchange posts, CC BY-SA license).

**How questions are picked:**
1. Keep question titles about ML topics (by tag), phrased as questions, 5–20 words long.
2. Remove coding and tool questions (Python errors, libraries, installation), since StudyMate teaches concepts.
3. Take a random sample of 120 (fixed seed, so the sample doesn't depend on StudyMate's performance).
4. For each question, list 5 possible lessons **in alphabetical order** (not ranked), so the labeler isn't pushed toward StudyMate's first choice.

**Labeling:** Our team labels each question with the correct lesson, or SKIP if no lesson fits. About 30 labeled questions are kept for the final test.

**Output:** `holdout_to_label.csv` and `lesson_titles.csv` in the evaluation folder.

In [42]:
# CODE CELL 8 — FINAL TEST, PART 1: PICK REAL QUESTIONS FOR LABELING
import re
from datasets import load_dataset

se = load_dataset("mgp123/datascience-stackexchange-with-similar-questions", split="train").to_pandas()
print("Stack Exchange rows:", len(se))

def tag_list(t):
    if t is None:
        return []
    return re.findall(r"[\w\-\.]+", t) if isinstance(t, str) else list(t)

ml_tags = {"machine-learning", "deep-learning", "neural-network", "classification", "regression",
           "clustering", "model-evaluations", "overfitting", "cross-validation", "nlp",
           "feature-selection", "feature-engineering", "preprocessing", "data-cleaning", "pca",
           "decision-trees", "random-forest", "logistic-regression", "linear-regression", "k-means",
           "gradient-descent", "transformer", "attention-mechanism", "word-embeddings",
           "transfer-learning", "hyperparameter-tuning", "accuracy", "metric", "confusion-matrix",
           "supervised-learning", "unsupervised-learning", "reinforcement-learning",
           "activation-function", "backpropagation", "dropout", "data-augmentation",
           "one-hot-encoding", "missing-data", "outlier", "generative-models", "bert", "tokenization"}
code_words = ["error", "keras", "tensorflow", "pytorch", "sklearn", "scikit", "pandas", "numpy",
              "python", " r ", "install", "code", "gpu", "spark", "sql", "excel", "matlab", "java"]
question_starts = ("what", "why", "how", "when", "which", "is ", "are ", "can ", "does ", "do ", "should ")

se["title"] = se["Title"].fillna("").str.strip()
words = se["title"].str.split().str.len()
is_ml = se["Tags"].apply(lambda t: bool(set(tag_list(t)) & ml_tags))
is_question = se["title"].str.lower().str.startswith(question_starts) | se["title"].str.endswith("?")
is_concept = ~se["title"].str.lower().apply(lambda s: any(w in f" {s} " for w in code_words))

pool = se[is_ml & is_question & is_concept & words.between(5, 20)].drop_duplicates("title")
print("Questions after filtering:", len(pool))

cand = pool.sample(n=min(120, len(pool)), random_state=42).reset_index(drop=True)

def five_suggestions(q):
    top5 = np.argsort(-hybrid_scores(q))[:5]
    return sorted(lib["title"].iloc[top5].tolist())      # alphabetical, not ranked

sheet = pd.DataFrame({"question_id": [f"H{i+1:03d}" for i in range(len(cand))],
                      "question": cand["title"]})
sugg = sheet["question"].apply(five_suggestions)
for k in range(5):
    sheet[f"suggestion_{k+1}"] = sugg.str[k]
sheet["expected_lesson"] = ""
sheet["also_acceptable"] = ""

eval_dir = os.path.join(project_path, "data", "evaluation")
sheet.to_csv(os.path.join(eval_dir, "holdout_to_label.csv"), index=False)
lib[["title", "topic"]].to_csv(os.path.join(eval_dir, "lesson_titles.csv"), index=False)
print(f"Saved {len(sheet)} questions to label, plus the list of {len(lib)} lesson titles.")
display(sheet.head(5))

Stack Exchange rows: 9172
Questions after filtering: 3188
Saved 120 questions to label, plus the list of 66 lesson titles.


,question_id,question,suggestion_1,suggestion_2,suggestion_3,suggestion_4,suggestion_5,expected_lesson,also_acceptable
0,H001,Gradient descent multidimensional linear regression - does learning rate affects concu...,Gradient Descent,Learning Rate,Linear Regression,Regularization,Transfer Learning,,
1,H002,Why is it bad to use the same test dataset over and over again?,Cross-Validation,Data Cleaning,Data Leakage,Overfitting,Train-Test Split,,
2,H003,Why use Gradient Descent when Gradient just solve the problem? (With Neural Nets),Activation Functions,Backpropagation,Gradient Descent,Loss and Optimizers,Neurons and Weights,,
3,H004,Ad click prediction: what are the negative examples?,Confusion Matrix,Features and Targets,R-Squared,Supervised Learning,Unsupervised Learning,,
4,H005,What is the significance of underflow during parameter update using stochastic gradien...,Backpropagation,Data Leakage,Gradient Descent,Learning Rate,Loss and Optimizers,,


## 9. Final Test, Part 2: Load and Check the Labels

**Purpose:** Load the labeled real questions, catch typos in lesson names, and show every question next to its label for a final review before the exam.

**Labeling:** Each of the 120 sampled questions was labeled with the lesson that answers it, or SKIP if no lesson really answers it (too advanced, too specific, or about code). Labels were drafted with AI assistance and reviewed and corrected by our team.

**Output:** Counts of labeled and skipped questions, any unknown lesson names, and the full list of labeled questions.

In [43]:
# CODE CELL 9 — FINAL TEST, PART 2: LOAD AND CHECK THE LABELS
eval_dir = os.path.join(project_path, "data", "evaluation")
labeled = pd.read_csv(os.path.join(eval_dir, "holdout_labeled.csv")).fillna("")
labeled["expected_lesson"] = labeled["expected_lesson"].astype(str).str.strip()
labeled["also_acceptable"] = labeled["also_acceptable"].astype(str).str.strip()

# Fix the one typo found in review, and save the corrected file
labeled["also_acceptable"] = labeled["also_acceptable"].str.replace("Embeeding", "Embedding")
labeled.to_csv(os.path.join(eval_dir, "holdout_labeled.csv"), index=False)

done = labeled[labeled["expected_lesson"] != ""]
skipped = done[done["expected_lesson"].str.upper() == "SKIP"]
holdout = done[done["expected_lesson"].str.upper() != "SKIP"].copy()

# Check: every lesson name must match a real title (catches typos)
titles = set(lib["title"])
backups = {b.strip() for s in holdout["also_acceptable"] for b in s.split(";") if b.strip()}
unknown = (set(holdout["expected_lesson"]) | backups) - titles

print(f"Reviewed: {len(done)} | Labeled: {len(holdout)} | SKIP: {len(skipped)} "
      f"({len(skipped) / len(done):.0%} of real questions are outside the library)")
print("Unknown lesson names (typos):", unknown if unknown else "none")

pd.set_option("display.max_colwidth", 90)
pd.set_option("display.max_rows", 80)
display(holdout[["question_id", "question", "expected_lesson", "also_acceptable"]])

Reviewed: 120 | Labeled: 53 | SKIP: 67 (56% of real questions are outside the library)
Unknown lesson names (typos): none


,question_id,question,expected_lesson,also_acceptable
0,H001,Gradient descent multidimensional linear regression - does learning rate affects concu...,Learning Rate,Gradient Descent
1,H002,Why is it bad to use the same test dataset over and over again?,Data Leakage,Overfitting
2,H003,Why use Gradient Descent when Gradient just solve the problem? (With Neural Nets),Gradient Descent,Loss and Optimizers
3,H004,Ad click prediction: what are the negative examples?,Features and Targets,
5,H006,What are differentiable modules used in deep learning,Activation Functions,Softmax Function
6,H007,What does discriminator of a GAN should do?,Generative AI,
7,H008,Is converting a numerical target to binary helpful?,Classification Basics,Features and Targets
9,H010,Why/When should I use VGG16 to do fine-tuning?,Fine-tuning,Transfer Learning
12,H013,Output a word instead of a vector after word embedding?,Embedding,Token
13,H014,How is the cross-product transformation defined for binary features?,Derived Features,


## 10. Final Exam on Real Questions (Run Once)

**Purpose:** Measure StudyMate on 53 real Data Science Stack Exchange questions it was never tuned on.

**What is tested:**
- **B. Two-stage search** (our final design) and **A. Hybrid only** (for comparison).
- **Off-topic guard:** how many real ML questions are wrongly blocked (same rule as Cell 5: block only if hybrid max < 0.18 AND reranker max < −9).

**Rule:** This cell is run once. No settings are changed after seeing these results; we only report them.

**Output:** Top-1, Recall@3, and lenient Top-1 for each option, the guard result, the remaining mistakes, and a saved results file.

In [44]:
# CODE CELL 10 — FINAL EXAM ON REAL QUESTIONS (RUN ONCE)
HYBRID_MIN, RERANK_MIN = 0.18, -9.0                      # same guard rule as Cell 5

def top3_hybrid(q):
    return lib["title"].iloc[np.argsort(-hybrid_scores(q))[:3]].tolist()

rows, blocked = [], 0
for _, q in holdout.iterrows():
    exp = q["expected_lesson"]
    ok = [exp] + [b.strip() for b in q["also_acceptable"].split(";") if b.strip()]
    two = search(q["question"])                          # B. two-stage
    hyb3 = top3_hybrid(q["question"])                    # A. hybrid only
    blocked += int(hybrid_scores(q["question"]).max() < HYBRID_MIN and two["rerank_score"].max() < RERANK_MIN)
    for name, top3 in [("A. Hybrid only", hyb3), ("B. Two-stage", two["title"].tolist())]:
        rows.append({"Option": name, "question_id": q["question_id"], "Question": q["question"],
                     "Expected": exp, "Predicted": top3[0],
                     "Top-1": int(top3[0] == exp), "Recall@3": int(exp in top3),
                     "Top-1 (lenient)": int(top3[0] in ok)})

final = pd.DataFrame(rows)
summary = final.groupby("Option").agg(**{
    "Top-1 %": ("Top-1", lambda x: round(x.mean() * 100, 1)),
    "Recall@3 %": ("Recall@3", lambda x: round(x.mean() * 100, 1)),
    "Top-1 lenient %": ("Top-1 (lenient)", lambda x: round(x.mean() * 100, 1)),
    "Top-1 correct": ("Top-1", "sum")})
print(f"=== FINAL EXAM: {len(holdout)} real questions ===")
display(summary)
print(f"\nGuard: real ML questions wrongly blocked = {blocked} of {len(holdout)}")

print("\nRemaining mistakes (B. Two-stage):")
display(final[(final["Option"] == "B. Two-stage") & (final["Top-1"] == 0)]
        [["question_id", "Question", "Expected", "Predicted", "Recall@3", "Top-1 (lenient)"]])

final.to_csv(os.path.join(eval_dir, "final_exam_results.csv"), index=False)
summary.to_csv(os.path.join(eval_dir, "final_exam_summary.csv"))
print("Saved: final_exam_results.csv and final_exam_summary.csv")

=== FINAL EXAM: 53 real questions ===


,Top-1 %,Recall@3 %,Top-1 lenient %,Top-1 correct
Option,,,,
A. Hybrid only,52.8,83.0,71.7,28
B. Two-stage,56.6,81.1,75.5,30



Guard: real ML questions wrongly blocked = 0 of 53

Remaining mistakes (B. Two-stage):


,question_id,Question,Expected,Predicted,Recall@3,Top-1 (lenient)
1,H001,Gradient descent multidimensional linear regression - does learning rate affects concu...,Learning Rate,Gradient Descent,1,1
3,H002,Why is it bad to use the same test dataset over and over again?,Data Leakage,Train-Test Split,0,0
7,H004,Ad click prediction: what are the negative examples?,Features and Targets,Supervised Learning,1,0
9,H006,What are differentiable modules used in deep learning,Activation Functions,Deep Learning,0,0
13,H008,Is converting a numerical target to binary helpful?,Classification Basics,Features and Targets,0,1
19,H014,How is the cross-product transformation defined for binary features?,Derived Features,Features and Targets,1,0
21,H017,What is an expert system,AI vs ML,Fine-tuning,0,0
25,H022,IS there any way we can add range in hyperparameter tuning of Decision Tree?,Hyperparameter,Model Improvement,1,1
27,H023,Does MLP always find local minimum,Gradient Descent,AI vs ML,0,0
33,H031,How to optimize XGBoost performance accuracy?,Model Improvement,Accuracy,1,0


Saved: final_exam_results.csv and final_exam_summary.csv
